In [ ]:
# 07 Error analysis and visualization (one explicit run)
#
# Question. Where does a chosen model err on the validation images, and (for C) does its uncertainty flag those errors?
# The run is chosen explicitly by EXPERIMENT_ID and RUN_ID; nothing is chosen for you. Validation predictions are checked against the run manifest, the frozen split and the training label file. No cell reads the official test set. Cell labels: [SAFE] | [READS RUN].
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ct_iqa.data.splits import load_manifest
from ct_iqa.evaluation.intervals import to_score_scale, uncertainty_error_correlation
from ct_iqa.experiment import report
from ct_iqa.experiment.runs import read_predictions, run_summary


def show(title, obj):
    """Print a section title, then the table as plain text."""
    print(f"\n=== {title} ===")
    print(obj.to_string())   # plain text table, no interactive viewer



In [ ]:
# 1. Choose and verify the run [READS RUN]
# Checks: completed status and artifact hashes, same frozen split, prediction filenames = the manifest's validation partition exactly, and labels identical to data/labels/ldct_iqac/train.json.
EXPERIMENT_ID = "A_reference_resnet50"      # or B_ranking_aware_resnet50 / C_uncertainty_aware_resnet50
RUN_ID = None

run_dir = None
if RUN_ID is None:
    print("Set RUN_ID to a completed run of", EXPERIMENT_ID, "- analysis PENDING.")
else:
    run_dir = Path("experiments/study_v2") / EXPERIMENT_ID / "runs" / RUN_ID
    S = run_summary(run_dir)                                   # raises on failed / incomplete / altered runs
    labels = json.loads(Path("data/labels/ldct_iqac/train.json").read_text())
    manifest = load_manifest(S["config"]["data"]["split_manifest"], labels)
    p = read_predictions(run_dir)
    checks = {
        "split manifest SHA-256 matches the run": manifest["manifest_sha256"] == S["manifest"]["split_manifest_sha256"],
        "predictions cover exactly the validation partition": sorted(p["filename"]) == sorted(manifest["validation"]),
        "no duplicate predictions": len(set(p["filename"])) == len(p["filename"]),
        "labels match the label file": all(abs(labels[n] - y) < 1e-9 for n, y in zip(p["filename"], p["label_0_4"])),
    }
    show(f"Verification of {run_dir.as_posix()}", pd.DataFrame({"passed": checks}))
    if not all(checks.values()):
        raise RuntimeError("predictions do not match the frozen validation partition; do not analyse this run")
    y, f = np.array(p["label_0_4"]), np.array(p["prediction_0_4"])
    r = f - y


In [ ]:
# 2. Residuals [READS RUN]
if run_dir:
    report.plot_prediction_scatter(y, f, f"{EXPERIMENT_ID}: prediction vs label (n = {len(y)})")
    plt.show()
    report.plot_residuals(y, f, f"{EXPERIMENT_ID}: residuals")
    plt.show()
    show("Overall error summary (score units)", pd.DataFrame({"value": {"n": len(r), "mean residual (bias)": r.mean(),
         "MAE": np.abs(r).mean(), "RMSE": np.sqrt((r ** 2).mean()), "max |error|": np.abs(r).max()}}).round(4))


In [ ]:
# 3. Error by label range [READS RUN]
# Counts are shown because ranges with few images give unstable summaries.
if run_dir:
    show("Error by label range, 1-point bins", report.error_by_score_bin(y, f).round(3))
    show("Error by label range, 0.5-point bins", report.error_by_score_bin(y, f, edges=tuple(np.arange(0, 4.01, 0.5))).round(3))


In [ ]:
# 4. Largest errors [READS RUN]
# Listed for inspection, not removed: every metric in this project is computed on all images.
TOP_K = 10
if run_dir:
    worst = np.argsort(-np.abs(r), kind="stable")[:TOP_K]
    show(f"{TOP_K} largest validation errors (all images remain in every metric)", pd.DataFrame({
        "filename": [p["filename"][i] for i in worst], "label": y[worst], "prediction": f[worst].round(3),
        "residual": r[worst].round(3), "|residual| / RMSE": (np.abs(r[worst]) / np.sqrt((r ** 2).mean())).round(2)}).set_index("filename"))


In [ ]:
# 5. Uncertainty vs error (Experiment C runs only) [READS RUN]
if run_dir and "log_var_normalized" in p:
    _, sigma = to_score_scale(p["prediction_normalized"], p["log_var_normalized"])
    report.plot_sigma_vs_error(sigma, np.abs(r))
    plt.show()
    print(f"Spearman(sigma, |error|) = {uncertainty_error_correlation(sigma, np.abs(r)):.3f} (n = {len(r)})")
    show("Binned calibration (mean sigma vs observed RMSE)", report.binned_sigma_calibration(sigma, r).round(3))
elif run_dir:
    print(f"{EXPERIMENT_ID} has no predicted uncertainty (not an Experiment C run).")


In [ ]:
# 6. Interval diagnostics (only when a calibration artifact exists) [READS RUN]
# Uses the q fitted on the calibration partition in calibration.json; nothing is re-fitted here.
import math

if run_dir and (run_dir / "calibration.json").is_file():
    cal = json.loads((run_dir / "calibration.json").read_text())
    q = math.inf if cal["quantile_q"] in ("inf", math.inf) else float(cal["quantile_q"])
    tables = report.calibration_frames(cal)
    show("Interval metrics on validation", tables["intervals"].round(4))
    report.plot_intervals(y, f, sigma, q, cal["target_coverage"], tables["terciles"], f"{RUN_ID}: validation intervals")
    plt.show()
elif run_dir:
    print("No calibration.json in this run: interval diagnostics are not applicable.")

# Limitations: image-level split without patient identifiers; 100 validation images (also used for checkpoint selection);
# labels are rater means on a 0.2 grid; uncertainty (C) is aleatoric only.
